# Model 4: GP with a linear mean, RQ kernel

**Purpose.** Fit one GC-GP model form to all seven properties and score it on a
single train/test split (80/20 unless the run plan sets `testSize`). This is one of the twelve models compared in the
kernel study (summarized in `03_Kernel_Comparison.ipynb`).

**Model form (method 4):** `y = GP(a·MW + b·y_GC + c, K(MW, y_GC))`

The GP learns the experimental value around a straight-line fit on both inputs. The line (a, b, c) is fitted together with the kernel hyperparameters.

**What this notebook produces** (in `Output_Data/kernel_comparison/<property>/model_4/`)

* `<property>_metrics.csv`: MAE, RMSE, R², MAPE and MPE on the training and test sets.
* `<property>_predictions.csv`: every prediction with its 95% interval.
* `<property>_GP_results.png` / `.pdf`: train, test and combined parity plots (600 dpi).
* `model_summary.txt`: fitted hyperparameters.

## 1. Environment and setup

In [ ]:
import sys
from pathlib import Path

# Find the repository root (the folder that contains src/gcgp) and make the
# shared code importable. This works whether Jupyter starts in notebooks/ or
# in the repository root.
repoRoot = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "gcgp").is_dir())
sys.path.insert(0, str(repoRoot / "src"))

import pandas as pd

from gcgp.settings import checkCuratedData, describeSetup, outputFolder, setGlobalSeed
from gcgp.singleSplit import compactSummary, runSingleSplitPlan
from gcgp.plotting import plotTestParityGrid
from gcgp.checkpoints import checkpointStatus

setGlobalSeed()
outputDir = outputFolder("kernel_comparison")
pd.set_option("display.precision", 3)
checkCuratedData()
print(describeSetup())
print(checkpointStatus())

**Checkpoints.** Every GP optimizer restart is saved in
`Output_Data/checkpoints/` as soon as it finishes. If the notebook stops (or is
rerun), finished restarts are loaded instead of refitted, so the run resumes
where it stopped and gives the same numbers. Set the environment variable
`GCGP_RESUME=0` to ignore the saved restarts and refit everything.

**Parallel restarts.** Restarts that still have to be fitted can run in
parallel worker processes: set the environment variable `GCGP_N_WORKERS`
(for example 4) before starting the kernel, or use
`bash scripts/run_notebooks.sh --workers 4`. The results are bit-for-bit the
same for any number of workers; `scripts/benchmark_gp_workers.py` shows which
number is fastest on your computer.

## 2. Data and model settings

Each property is read from its curated dataset in `Output_Data/curated_data/`
(written by `00_Data_Curation.ipynb`). The inputs are molecular weight and the
GC estimate; the target is the CRC value.

Every entry of the run plan below lists the properties it applies to and the GP
settings used for them:

* `kernel`: covariance function (RQ, RBF, or a Matérn kernel).
* `anisotropic`: whether each input gets its own length scale.
* `optMethod`: SciPy optimizer for the hyperparameters.
* `trainLikelihood`: whether the noise term is fitted (if False it stays at 1e-5).
* `retrainGp`: number of random restarts; the best converged restart is kept.
* `testSize` (optional): fraction of molecules held out for testing; 0.2 when not given.

**Notes for this model.** Same settings for every property. The original notebook fitted Hvap twice with identical settings; the duplicate run is removed.

In [ ]:
methodNumber = 4
saveName = "model_4"

runPlan = [
    {"properties": ["Tb", "Tm", "Pc", "Vc", "Tc", "Hvap", "logP"],
     "kernel": "RQ", "anisotropic": False, "optMethod": "L-BFGS-B",
     "trainLikelihood": False, "retrainGp": 10},
]

pd.DataFrame(runPlan)

## 3. Fit and evaluate

For each property: split the molecules into training and test sets (seed 42), fit the GP with several
restarts, and predict both sets. The per-property figures are saved to disk
rather than shown here, to keep the notebook short; Section 4 shows an overview.

In [ ]:
summary = runSingleSplitPlan(runPlan, methodNumber, saveName, outputDir, showPlot=False)
summary.to_csv(outputDir / f"{saveName}_summary.csv", index=False)

## 4. Results

### 4.1 Metrics for every run

`Test_R2`, `Test_RMSE` and `Test_MAE` are measured on the 20% of molecules the
GP never saw. `CI_Test_Outside` is the fraction of test molecules whose
experimental value falls outside the predicted 95% interval; a well-calibrated
model should be close to 0.05.

In [ ]:
compactSummary(summary)

### 4.2 Test-set parity plots

Each panel shows the held-out molecules for one property. The error bars are the
95% prediction intervals and the dashed line is perfect agreement.

In [ ]:
tables = {code: pd.read_csv(outputDir / code / saveName / f"{code}_predictions.csv")
          for code in summary["Property"]}
if tables:
    plotTestParityGrid(tables, f"Model {4}, test set",
                       savePath=outputDir / f"{saveName}_test_parity_overview.png")

## 5. Summary of the results

The text below is written from the metrics of this run and saved as
`results_summary.md` in this model's output folder. A run that could not be
fitted (for example when no optimizer restart converged) is reported and left
out instead of stopping the notebook. The metrics files are read by
`03_Kernel_Comparison.ipynb`, which puts all twelve models side by side.

In [ ]:
from gcgp.narrative import describeSingleSplit, showAndSave

showAndSave(describeSingleSplit(summary, "Model 4: GP with a linear mean, RQ kernel"),
            outputDir / f"{saveName}_results_summary.md");